In [13]:
import json

import pandas as pd
import numpy as np

from datasets import load_dataset
import transformers
from transformers import AutoTokenizer

In [14]:

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("nvidia/HelpSteer3", "preference")

In [15]:
ds['train'][0].keys()

dict_keys(['domain', 'language', 'context', 'response1', 'response2', 'overall_preference', 'individual_preference'])

In [16]:
filtered = ds.filter(lambda entry: entry['overall_preference'] != 0)


In [17]:
def to_chosen_rejected(entry):
    context = entry['context']
    if entry['overall_preference'] > 0:
        chosen = entry['response2']
        rejected = entry['response1']
    else:
        chosen = entry['response1']
        rejected = entry['response2']
    entry['chosen'] = context + [{"role": "assistant", "content": chosen}]
    entry['rejected'] = context + [{"role": "assistant", "content": rejected}]
    entry['preference_strength'] = abs(entry['overall_preference'])
    return entry

preference_ds = ds.map(to_chosen_rejected)

In [6]:
preference_ds.push_to_hub('ktolnos/helpsteer3-preference-chosenrrejected')

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/20 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/181M [00:00<?, ?B/s]

Creating parquet from Arrow format:   0%|          | 0/20 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/166M [00:00<?, ?B/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/18.0M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/ktolnos/helpsteer3-preference-chosenrrejected/commit/85a90c51df5ce2152c227ddb01aa9deef6fb9d77', commit_message='Upload dataset', commit_description='', oid='85a90c51df5ce2152c227ddb01aa9deef6fb9d77', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/ktolnos/helpsteer3-preference-chosenrrejected', endpoint='https://huggingface.co', repo_type='dataset', repo_id='ktolnos/helpsteer3-preference-chosenrrejected'), pr_revision=None, pr_num=None)

In [18]:
tenk_ds = preference_ds.take(10_000)
# tenk_ds.push_to_hub('ktolnos/helpsteer3-preference-chosenrrejected-10k')

AttributeError: 'DatasetDict' object has no attribute 'take'

In [7]:
old_ds = load_dataset('gagan3012/helpsteer2-preference-v2')
old_ds['train'][0].keys()

dict_keys(['preference_strength', 'chosen', 'rejected'])

In [8]:
load_dataset('ktolnos/helpsteer3-preference-chosenrrejected')

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/181M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/18.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/38459 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2017 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['domain', 'language', 'context', 'response1', 'response2', 'overall_preference', 'individual_preference', 'chosen', 'rejected', 'preference_strength'],
        num_rows: 38459
    })
    validation: Dataset({
        features: ['domain', 'language', 'context', 'response1', 'response2', 'overall_preference', 'individual_preference', 'chosen', 'rejected', 'preference_strength'],
        num_rows: 2017
    })
})

In [9]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
tokenizer.max_length = 2048

In [10]:
chosen_messages = preference_ds['train'][0]['chosen']
chosen_messages

[{'content': 'lets do python coding', 'role': 'user'},
 {'content': ' def greet(name):\n      return f"Hello, {name}!"\n\nprint(greet("Alice"))\n# Output: Hello, Alice!\n\ndef add(a, b):\n    return a + b\n\nprint(add(3, 4))\n# Output: 7\n\ndef is_even(number):\n    return number % 2 == 0\n\nprint(is_even(5))\n# Output: False\n\ndef factorial(n):\n    if n == 0:\n        return 1\n    else:\n        return n * factorial(n-1)\n\nprint(factorial(5))\n# Output: 120\n```',
  'role': 'assistant'},
 {'content': 'First of all, in this scenario, two programmers are collaborating. Programmer A can do as much work as a per day, and programmer B can do as much work as b. Two programmers always finish n (=a+b) jobs a day. The first input given are three numbers, n a b respectively, separated by spaces. So make the code that gets the three inputs.\n',
  'role': 'user'},
 {'content': "7 2\n\nHere's the Python code that solves the problem:\n\n```python\ndef work_rate(a, b):\n    return a / (a + b)\n\

In [11]:
preference_ds['train'][0]['context']

[{'role': 'user', 'content': 'lets do python coding'},
 {'role': 'assistant',
  'content': ' def greet(name):\n      return f"Hello, {name}!"\n\nprint(greet("Alice"))\n# Output: Hello, Alice!\n\ndef add(a, b):\n    return a + b\n\nprint(add(3, 4))\n# Output: 7\n\ndef is_even(number):\n    return number % 2 == 0\n\nprint(is_even(5))\n# Output: False\n\ndef factorial(n):\n    if n == 0:\n        return 1\n    else:\n        return n * factorial(n-1)\n\nprint(factorial(5))\n# Output: 120\n```'},
 {'role': 'user',
  'content': 'First of all, in this scenario, two programmers are collaborating. Programmer A can do as much work as a per day, and programmer B can do as much work as b. Two programmers always finish n (=a+b) jobs a day. The first input given are three numbers, n a b respectively, separated by spaces. So make the code that gets the three inputs.\n'},
 {'role': 'assistant',
  'content': "7 2\n\nHere's the Python code that solves the problem:\n\n```python\ndef work_rate(a, b):\n  

In [12]:
tokenizer.apply_chat_template(chosen_messages, tokenize=False, add_generation_prompt=True, enable_thinking=False, truncation=True, max_length=tokenizer.max_length)

'<|im_start|>user\nlets do python coding<|im_end|>\n<|im_start|>assistant\n def greet(name):\n      return f"Hello, {name}!"\n\nprint(greet("Alice"))\n# Output: Hello, Alice!\n\ndef add(a, b):\n    return a + b\n\nprint(add(3, 4))\n# Output: 7\n\ndef is_even(number):\n    return number % 2 == 0\n\nprint(is_even(5))\n# Output: False\n\ndef factorial(n):\n    if n == 0:\n        return 1\n    else:\n        return n * factorial(n-1)\n\nprint(factorial(5))\n# Output: 120\n```<|im_end|>\n<|im_start|>user\nFirst of all, in this scenario, two programmers are collaborating. Programmer A can do as much work as a per day, and programmer B can do as much work as b. Two programmers always finish n (=a+b) jobs a day. The first input given are three numbers, n a b respectively, separated by spaces. So make the code that gets the three inputs.\n<|im_end|>\n<|im_start|>assistant\n7 2\n\nHere\'s the Python code that solves the problem:\n\n```python\ndef work_rate(a, b):\n    return a / (a + b)\n\ndef 